In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, random_split, WeightedRandomSampler
from torchvision import transforms, datasets
import timm
from tqdm import tqdm
import json
from datetime import datetime
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

/home/Abo/miniconda3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Run this script ONCE to preprocess and align all faces in your datasets.

import os
import cv2
import numpy as np
import mediapipe as mp
from PIL import Image
from pathlib import Path
from tqdm import tqdm

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

def align_face_mediapipe(img):
    """Face alignment using MediaPipe landmarks"""
    # Convert to RGB if needed
    if isinstance(img, np.ndarray):
        if len(img.shape) == 2:
            rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        rgb = np.array(img.convert('RGB'))
    
    results = face_mesh.process(rgb)
    
    if not results.multi_face_landmarks:
        return None
    
    h, w = rgb.shape[:2]
    landmarks = results.multi_face_landmarks[0]
    
    # Convert normalized landmarks to pixel coordinates
    points = np.array([[lm.x * w, lm.y * h] for lm in landmarks.landmark])
    
    # Get eye centers
    left_eye_indices = [33, 133, 145, 153, 154, 155, 157, 158, 159, 160, 161, 163]
    right_eye_indices = [263, 362, 373, 380, 381, 382, 384, 385, 386, 387, 388, 390]
    
    left_eye = points[left_eye_indices].mean(axis=0)
    right_eye = points[right_eye_indices].mean(axis=0)
    
    # Calculate rotation angle
    dY = right_eye[1] - left_eye[1]
    dX = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dY, dX))
    
    eye_center = (float((left_eye[0] + right_eye[0]) / 2), 
                  float((left_eye[1] + right_eye[1]) / 2))
    
    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    
    rotated = cv2.warpAffine(rgb, M, (w, h), flags=cv2.INTER_CUBIC)
    
    ones = np.ones(shape=(len(points), 1))
    points_ones = np.hstack([points, ones])
    rotated_points = M.dot(points_ones.T).T
    
    left_face = int(rotated_points[234][0])
    right_face = int(rotated_points[454][0])
    chin = int(rotated_points[152][1])
    
    eye_center_rotated = M.dot([eye_center[0], eye_center[1], 1])
    face_height = chin - eye_center_rotated[1]
    top = int(eye_center_rotated[1] - face_height * 0.5)
    
    # Ensure boundaries are within image
    top = max(0, top)
    left_face = max(0, left_face)
    chin = min(h, chin)
    right_face = min(w, right_face)
    
    # Crop face
    if right_face > left_face and chin > top:
        cropped = rotated[top:chin, left_face:right_face]
        aligned = cv2.resize(cropped, (128, 128))
        return Image.fromarray(aligned)
    
    return None


def preprocess_dataset(input_dir, output_dir, target_size=128):
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    
    if not input_path.exists():
        print(f"Input directory does not exist: {input_dir}")
        return
    
    classes = [d.name for d in input_path.iterdir() if d.is_dir()]
    
    print(f"Processing: {input_dir}")
    print(f"Output to: {output_dir}")
    print(f"Found {len(classes)} emotion classes: {classes}")
    
    total_processed = 0
    total_failed = 0
    
    for emotion_class in classes:
        print(f"\nProcessing class: {emotion_class}")
        
        class_output_dir = output_path / emotion_class
        class_output_dir.mkdir(parents=True, exist_ok=True)
        
        class_input_dir = input_path / emotion_class
        image_files = list(class_input_dir.glob('*.jpg')) + \
                      list(class_input_dir.glob('*.png')) + \
                      list(class_input_dir.glob('*.jpeg'))
        
        if len(image_files) == 0:
            print(f"No images found in {class_input_dir}")
            continue
        
        for img_file in tqdm(image_files, desc=f"  Aligning {emotion_class}"):
            try:
                # Read image
                img = cv2.imread(str(img_file))
                if img is None:
                    total_failed += 1
                    continue
                
                # Align face
                aligned = align_face_mediapipe(img)
                
                output_file = class_output_dir / img_file.name
                
                if aligned is not None:
                    aligned.save(output_file)
                    total_processed += 1
                else:
                    # Fallback: save resized original (center crop)
                    img_pil = Image.open(img_file)
                    min_dim = min(img_pil.size)
                    left = (img_pil.width - min_dim) // 2
                    top = (img_pil.height - min_dim) // 2
                    img_cropped = img_pil.crop((left, top, left + min_dim, top + min_dim))
                    resized = img_cropped.resize((target_size, target_size))
                    resized.save(output_file)
                    total_failed += 1
                    
            except Exception as e:
                print(f"    Error processing {img_file.name}: {e}")
                total_failed += 1
    
    print(f"   Successfully aligned: {total_processed}")
    print(f"   Failed/Fallback: {total_failed}")
    print(f"   Total images: {total_processed + total_failed}")

datasets_to_process = [
    ("./data/FER-2013/train", "./data/FER-2013-aligned/train"),
    ("./data/FER-2013/test", "./data/FER-2013-aligned/test"),
    ("./data/AffectNet/train", "./data/AffectNet-aligned/train"),
    ("./data/AffectNet/test", "./data/AffectNet-aligned/test"),
]
    
for input_dir, output_dir in datasets_to_process:
    preprocess_dataset(input_dir, output_dir, target_size=128)

face_mesh.close()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2025-11-16 17:40:27.199964: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-16 17:40:27.390953: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1763295029.153714   94219 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1763295029.158707   94399 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.6-arch1.1), renderer: AMD Radeon 610M (radeonsi, raphael_mendocino, LLVM 21.1.4, DRM 3.64, 6.17.7-arch1-1)
INFO: Created TensorFlow Lite XNNPACK delegate f

Processing: ./data/FER-2013/train
Output to: ./data/FER-2013-aligned/train
Found 7 emotion classes: ['sad', 'anger', 'happy', 'disgust', 'neutral', 'fear', 'surprise']

Processing class: sad


  Aligning sad:   0%|          | 0/4830 [00:00<?, ?it/s]W0000 00:00:1763295029.179487   94366 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763295029.186670   94374 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
  Aligning sad:  21%|██        | 991/4830 [00:05<00:22, 170.69it/s]


KeyboardInterrupt: 

In [6]:
FER_DATA_DIR = "./data/FER-2013-aligned"
AFFECT_DATA_DIR = "./data/AffectNet-aligned"
OUTPUT_DIR = "./outputs"

MODEL_NAME = "vit_small_patch16_224"
IMG_SIZE = 128
NUM_CLASSES = 7

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

VAL_FRACTION = 0.5
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
EARLY_STOP_PATIENCE = 7

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Dataset loading and preprocessing
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3)
    ], p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

fer_train = datasets.ImageFolder(os.path.join(FER_DATA_DIR, "train"), transform=train_transform)
fer_test = datasets.ImageFolder(os.path.join(FER_DATA_DIR, "test"), transform=val_transform)

affect_train = datasets.ImageFolder(os.path.join(AFFECT_DATA_DIR, "train"), transform=train_transform)
affect_test = datasets.ImageFolder(os.path.join(AFFECT_DATA_DIR, "test"), transform=val_transform)

train_dataset = ConcatDataset([fer_train, affect_train])
test_dataset = ConcatDataset([fer_test, affect_test])

total_test = len(test_dataset)
val_size = int(VAL_FRACTION * total_test)
test_size = total_test - val_size
val_dataset, test_dataset = random_split(test_dataset, [val_size, test_size], 
                                         generator=torch.Generator().manual_seed(42))

emotion_labels = fer_train.classes
print(f"\nEmotions: {emotion_labels}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Get all training labels
train_labels = []
for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    train_labels.append(label)
train_labels = np.array(train_labels)

print("\nTraining set:")
for i, emotion in enumerate(emotion_labels):
    count = np.sum(train_labels == i)
    percentage = 100 * count / len(train_labels)
    print(f"{emotion:10s}: {count:6d} samples ({percentage:5.2f}%)")

# Class Weights Consideration
class_weights = None
class_weights = compute_class_weight('balanced', 
                                     classes=np.arange(NUM_CLASSES), 
                                     y=train_labels)
class_weights = torch.FloatTensor(class_weights)
    
print("\nClass weights:")
for i, (emotion, weight) in enumerate(zip(emotion_labels, class_weights)):
    print(f"{emotion:10s}: {weight:.4f}")

train_sampler = None
shuffle_train = True

# Balanced Sampler Consideration
class_counts = np.bincount(train_labels)
weights = 1.0 / class_counts[train_labels]
train_sampler = WeightedRandomSampler(weights=weights, 
                                     num_samples=len(weights), 
                                     replacement=True)
shuffle_train = False

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                          shuffle=shuffle_train, sampler=train_sampler,
                          num_workers=4, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                       shuffle=False, num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                        shuffle=False, num_workers=4, pin_memory=True)

Using device: cuda

Emotions: ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train: 43258 | Val: 10192 | Test: 10192

Training set:
anger     :   5495 samples (12.70%)
disgust   :   1665 samples ( 3.85%)
fear      :   5609 samples (12.97%)
happy     :   9555 samples (22.09%)
neutral   :   7723 samples (17.85%)
sad       :   7921 samples (18.31%)
surprise  :   5290 samples (12.23%)

Class weights:
anger     : 1.1246
disgust   : 3.7115
fear      : 1.1017
happy     : 0.6468
neutral   : 0.8002
sad       : 0.7802
surprise  : 1.1682


In [7]:
# Model

model = timm.create_model(MODEL_NAME, pretrained=True, 
                         img_size=IMG_SIZE, in_chans=1, num_classes=NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.2f} MB")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                 factor=SCHEDULER_FACTOR, 
                                                 patience=SCHEDULER_PATIENCE, 
                                                 verbose=True)

Total parameters: 21,421,063
Model size: ~85.68 MB


/home/Abo/miniconda3/envs/pytorch/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [ ]:
# Create run directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = os.path.join(OUTPUT_DIR, f"run_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

best_val_acc = 0.0
best_model_path = os.path.join(run_dir, "best_model.pth")
patience_counter = 0


# Training

for epoch in range(EPOCHS):
    
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 
                         'acc': f'{100*train_correct/train_total:.2f}%'})
    
    train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validating")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 
                             'acc': f'{100*val_correct/val_total:.2f}%'})
    
    val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print summary
    print(f"\nEpoch {epoch+1:02d} Summary:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Learning Rate: {current_lr:.6f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered! No improvement for {EARLY_STOP_PATIENCE} epochs")
        break

print("Training done")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

with open(os.path.join(run_dir, "training_history.json"), 'w') as f:
    json.dump(history, f, indent=4)

# Plot training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', linewidth=2)
ax2.plot(history['val_acc'], label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(run_dir, 'training_curves.png'), dpi=300)
plt.close()

Using device: cuda

Emotions: ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Train: 43258 | Val: 10192 | Test: 10192


KeyboardInterrupt: 

In [ ]:
# Testing

# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_loss = test_loss / len(test_loader)
test_acc = 100 * test_correct / test_total

print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")


all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Collecting predictions"):
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("\nClassification Report:")
report = classification_report(all_labels, all_preds, 
                               target_names=emotion_labels, digits=4)
print(report)

with open(os.path.join(run_dir, "classification_report.txt"), 'w') as f:
    f.write(report)

print("\nPer-Class Accuracy:")
for i, emotion in enumerate(emotion_labels):
    class_mask = all_labels == i
    if class_mask.sum() > 0:
        class_acc = (all_preds[class_mask] == all_labels[class_mask]).mean() * 100
        print(f"{emotion:10s}: {class_acc:.2f}%")

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12, fontweight='bold')
plt.ylabel('True', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(run_dir, 'confusion_matrix.png'), dpi=300)
plt.close()